In [14]:
# Slice Dimension Attributes defined in the plugin. Please check all queries and replace <KEY HERE> with a valid name.
# For example: If slice is defined by Version.[Version Name] and Time.[Month]
# input_df = ibpl Select ([Version].[Version Name].[<KEY HERE>] * [Time].[Month].[<KEY HERE>] * [Item].[Item Number]) on row, ({Measure.[M1], Measure.[M2]}) on column limit 5000;
#                             update <KEY HERE> to valid names
# input_df = ibpl Select ([Version].[Version Name].[CurrentWorkingView] * [Time].[Month].[January] * [Item].[Item Number]) on row, ({Measure.[M1], Measure.[M2]}) on column limit 5000;

_sales_df = "select (Version.[Version Name]*Product.[Product].[196426]*Time.FiscalWeek*SalesAccount.[Account]*Location.[Location]*{Measure.[DPSellOutUnitsActuals],Measure.[Mean Pricing Save PCT],Measure.[Placement Count],Measure.[Promotion Count],Measure.[DPSellOutPrice]});"
_DPBaseInputs = "select (Version.[Version Name]*Product.[Product].[196426]*Time.FiscalWeek*SalesAccount.[Account]*Location.[Location]*{Measure.[DPSellOutUnitsActuals],Measure.[Mean Pricing Save PCT],Measure.[Placement Count],Measure.[Promotion Count],Measure.[DPSellOutPrice]});"


# Initialize the O9DataLake with the input parameters and dataframes
# Data can be accessed with O9DataLake.get(<Input Name>)
# Overwritten values will not be reflected in the O9DataLake after initialization

from o9_common_utils.O9DataLake import O9DataLake, ResourceType, DataSource,PluginSetting
sales_df = O9DataLake.register("sales_df",data_source = DataSource.LS, entity_type = ResourceType.IBPL, query = _sales_df,plugin_setting = PluginSetting.Inputs, spark = spark)
DPBaseInputs = O9DataLake.register("DPBaseInputs",data_source = DataSource.LS, entity_type = ResourceType.IBPL, query = _DPBaseInputs,plugin_setting = PluginSetting.Inputs ,spark = spark)
O9DataLake.register("Version.[Version Name]", data_source = DataSource.LS, entity_type = ResourceType.IBPL,plugin_setting = PluginSetting.SliceDimension, spark = spark)

O9DataLake.register("out_df1",data_source = DataSource.LS,entity_type = ResourceType.IBPL,plugin_setting = PluginSetting.Outputs)
script_params = O9DataLake.register({}, data_source = DataSource.LS,plugin_setting = PluginSetting.ScriptParam)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [15]:
type(sales_df)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

<class 'pyspark.sql.dataframe.DataFrame'>

In [16]:
sales_df.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------------+--------------+--------------+-------------------+----------------+---------------------+------------------+--------------+--------------+--------------+
|VersionVersionName|ProductProduct|TimeFiscalWeek|SalesAccountAccount|LocationLocation|DPSellOutUnitsActuals|MeanPricingSavePCT|PlacementCount|PromotionCount|DPSellOutPrice|
+------------------+--------------+--------------+-------------------+----------------+---------------------+------------------+--------------+--------------+--------------+
|                S1|        196426|      W03-2016|                ALL|             ALL|                  1.0|               NaN|           NaN|           NaN|           6.0|
|CurrentWorkingView|        196426|      W03-2016|                ALL|             ALL|                  1.0|               NaN|           NaN|           NaN|           6.0|
|                S1|        196426|      W05-2016|                ALL|             ALL|                  2.0|               NaN|  

In [17]:
#getting input from dataLake
sales = O9DataLake.get('sales_df')
dpbaseinputs = O9DataLake.get('DPBaseInputs')
import logging
out_df1 = None
logger = logging.getLogger('o9_logger')


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…